# 09 · Metal SMILES

rxembed writes a metal complex as a dative SMILES, with donor-to-metal arrows and formal charges on the ligands.
`rx.smiles` writes its connectivity, and `rx.cxsmiles` adds the metal's arrangement as CX notes, so one string names
one species and `rx.embed` reads it back.

In [ ]:
import rxembed as rx
import xyzrender

## Covalent and dative input

A covalent metal SMILES and its ionic dative form are one species: `rx.smiles` writes each pair below as the same
dative string.

In [ ]:
FORMS = {  # covalent, dative
    "chloride": ("N->[Pt](<-N)(Cl)Cl", "N->[Pt+2](<-N)(<-[Cl-])<-[Cl-]"),
    "phosphine and anion": ("CP(C)(C)->[Rh]Cl", "CP(C)(C)->[Rh+]<-[Cl-]"),
    "terminal oxo": ("O=[V](Cl)(Cl)Cl", "[O-2]->[V+5](<-[Cl-])(<-[Cl-])<-[Cl-]"),
}
for kind, pair in FORMS.items():
    print(f"{kind:20s}", {rx.smiles(rx.parse_smiles(text)) for text in pair})

## Connectivity and arrangement

Cis- and trans-platin share one SMILES. `rx.cxsmiles` appends an `atomProp` block: `atomNote.SPL` marks the
square-planar metal and `atomNote.s0` to `s3` seat each donor on a vertex, so the ammines sit on neighbouring vertices
in cis and opposite ones in trans. RDKit keeps the notes and rxembed reads them.

In [ ]:
for platin in rx.isomers("N->[Pt+2](<-N)(<-[Cl-])<-[Cl-]", geometry="SPL"):
    print(rx.identity(platin).labels[0], rx.smiles(platin))
    print("     ", rx.cxsmiles(platin))

## From a geometry and back

A geometry writes the same way. This Ru pincer's η2-bound carbamoyl C=O shares the slot `s1`, since a side-on face is
one vertex. Embedding the string rebuilds that species, and `rx.cxsmiles` of the result is the same string.

In [ ]:
ru = rx.read_xyz("structures/ru-co.xyz", charge=1)
written = rx.cxsmiles(ru)
print(written)

rebuilt = rx.embed(written, n=2, seed=1).minimize()
print(rebuilt)
print("same string:", rx.cxsmiles(rebuilt.mol) == written)
xyzrender.render(xyzrender.load(rebuilt.lowest(1).mol))

## RDKit's stereo tags

RDKit's `@SP`, `@TB` and `@OH` tags state an arrangement through the order of the metal's neighbours. `rx.isomers`
reads each into the one species it names: `@SP1` lists the chlorides cis and `@SP2` trans, and `@OH1` and `@OH2` are
the two hands of one octahedral arrangement.

In [ ]:
TAGGED = {
    "@SP1": "[Cl-]->[Pt@SP1+2](<-[Cl-])(<-[NH3])<-[NH3]",
    "@SP2": "[Cl-]->[Pt@SP2+2](<-[Cl-])(<-[NH3])<-[NH3]",
    "@TB1": "[Cl-]->[Fe@TB1+2](<-[Cl-])(<-[Br-])(<-[Br-])<-[NH3]",
    "@OH1": "[F-]->[Pt@OH1+4](<-[F-])(<-[Cl-])(<-[Cl-])(<-[Br-])<-[Br-]",
    "@OH2": "[F-]->[Pt@OH2+4](<-[F-])(<-[Cl-])(<-[Cl-])(<-[Br-])<-[Br-]",
}
for tag, smiles in TAGGED.items():
    print(tag, rx.isomers(smiles).summary())

## Embedding a tagged species

RDKit's own embedding does not reliably enforce these tags ([RDKit Book](https://rdkit.org/docs/RDKit_Book.html#support-for-non-tetrahedral-atomic-stereochemistry)).
`rx.embed` builds the species the tag names and checks the metal's hand after relaxation, so every conformer writes
the CX string of the `@OH1` species.

In [ ]:
stated = rx.cxsmiles(rx.isomers(TAGGED["@OH1"])[0])
octahedra = rx.embed(TAGGED["@OH1"], n=4, seed=1).minimize()
print(octahedra)
print("every conformer the stated species:", all(rx.cxsmiles(octahedra[i].mol) == stated for i in range(octahedra.n)))
xyzrender.render(xyzrender.load(octahedra.lowest(1).mol))